In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import ipywidgets as widgets
from IPython.display import display, clear_output
import matplotlib.gridspec as gridspec
from sklearn.preprocessing import StandardScaler, FunctionTransformer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.manifold import TSNE
from sklearn.cluster import DBSCAN
from sklearn.mixture import GaussianMixture



In [ ]:
df = pd.read_csv('CC GENERAL.csv')

In [ ]:
df.head()

In [ ]:
df.info()

###Droping Customer ID, Negligible Info

In [ ]:
df.drop(columns=['CUST_ID'],inplace=True)

In [ ]:
df.head()

In [ ]:
df.duplicated().sum()

###Checking NaN Values

In [ ]:
df.isna().sum()

##Let's check why do we have 313 Nans on minimum payments.

In [ ]:
df[df['MINIMUM_PAYMENTS'].isna()]


### For most records where `MINIMUM_PAYMENTS` is NaN, the `PAYMENTS` value is 0. This suggests that these credit card users are not making any payments — either they are accumulating debt or they are inactive users with zero balance.

In [ ]:
df[(df['MINIMUM_PAYMENTS'].isna()) & (df['PAYMENTS'] == 0)].count()[0]


###We can put the `MINIMUM_PAYMENTS` for those with zero balance = 0, and fill the rest with median value..

In [ ]:
df[(df['MINIMUM_PAYMENTS'].isna()) & (df['PAYMENTS'] == 0)] = 0

In [ ]:
df.isna().sum()

In [ ]:
df[df['MINIMUM_PAYMENTS'].isna()]
#seeing the rest of 73 nan vals for minimum payment

In [ ]:
plt.hist(df['MINIMUM_PAYMENTS'],bins = 50)


###Minimum payments is right skewed, most values are low.

In [ ]:
df['MINIMUM_PAYMENTS'].median()

#We'll drop the column where `CREDIT_LIMIT` is NaN, it's one record anyways. Probably data entry problem

In [ ]:
df.loc[df['CREDIT_LIMIT'].isna()]

In [ ]:
df.drop(5203, inplace=True)

In [ ]:
#Filling rest with median

df['MINIMUM_PAYMENTS']= df['MINIMUM_PAYMENTS'].fillna(df['MINIMUM_PAYMENTS'].median())

#Handled all NaNs Successfully.

In [ ]:
df.isna().sum()

#Let's now Explore the Data.

The `BALANCE` feature is right-skewed: most customers carry relatively low debt, while a smaller number have high balances. Some of these high-balance customers may simply be wealthier and capable of managing larger debts.

In [ ]:
plt.hist(df['BALANCE'],bins = 10)

#Same thing with `CREDIT_LIMIT`, this is representative of incomes of people. some people are rich with high credit limit, others are not. some people are of medium standard of living.

In [ ]:
plt.hist(df['CREDIT_LIMIT'], bins=10)

#How long have been the users holding their credit card?

In [ ]:
sns.countplot(data=df,x='TENURE')

#We can see that some credit card holders are brand new with 0 month tenure, this data was collected for last 6month behavior.. let's see these 0month tenure users! Probably not enough data collected for their behavior

In [ ]:
df[df['TENURE']==0]

#All users with tenure 0 are FULL Zero Rows. I'll just drop these 240 rows. They give 0 insight about user behavior. just dummy data.

In [ ]:
df = df[df['TENURE'] != 0]


#This made me check if there are other full zero rows. But there are none.

In [ ]:
numeric_cols = df.select_dtypes(include='number').columns
full_zero_rows = df[(df[numeric_cols] == 0).all(axis=1)]
print(f"Number of full-zero rows: {full_zero_rows.shape[0]}")
full_zero_rows


#Purchases are right skewed.
#Most customemrs have low or medium spending.

In [ ]:
plt.hist(df['PURCHASES'],bins=10)


#Right skewed. same.


In [ ]:
plt.hist(df['CASH_ADVANCE'],bins=10)

#We have like 2 distributions here. we have a significant amount of people with low frequency, and also significant amount of people with high frequency, its multimodal

In [ ]:
plt.hist(df['PURCHASES_FREQUENCY'])

In [ ]:
df.hist(figsize=(20,20))


- PURCHASES_FREQUENCY is strongly correlated with PURCHASES_INSTALLMENTS_FREQUENCY (0.86), indicating consistent purchasing behavior patterns.

- CASH_ADVANCE_FREQUENCY and CASH_ADVANCE_TRX are highly correlated (0.80), meaning transaction count largely reflects frequency.

- Cash advance variables show weak or negative correlation with purchase variables → suggests different customer behavior segments.

- CREDIT_LIMIT moderately correlates with BALANCE (0.54) and PAYMENTS (0.43), indicating higher-limit customers tend to spend and pay more.

- MINIMUM_PAYMENTS has weak correlations overall → may represent a distinct financial behavior group (e.g., revolving customers).

In [ ]:
corr_matrix = df.corr()

plt.figure(figsize=(15, 15)) # Adjust figure size
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt=".2f")
plt.title('Correlation Matrix Heatmap')
plt.show()

#more corr with balance (debts) is with cash advance, because when u take cash. more interest. more debts.
#same with balance vs credit limit. more money more debt. trivial
#Rest of high correlations are obvious why. more purchases more purchase transactions etc


#Let's log transform the unbounded right skewed distributions [payments,credit_limit,minimum_payments,cash_advance,installments_purchases,purchases,oneoff_purchases,balance]

In [ ]:



right_skewed = [
    'PAYMENTS', 'CREDIT_LIMIT', 'MINIMUM_PAYMENTS', 'CASH_ADVANCE',
    'INSTALLMENTS_PURCHASES', 'PURCHASES', 'ONEOFF_PURCHASES', 'BALANCE','PURCHASES_TRX','CASH_ADVANCE_TRX',
]

bounded_features = [
    'BALANCE_FREQUENCY', 'PURCHASES_FREQUENCY', 'ONEOFF_PURCHASES_FREQUENCY',
    'PURCHASES_INSTALLMENTS_FREQUENCY', 'CASH_ADVANCE_FREQUENCY', 'PRC_FULL_PAYMENT','TENURE'
]

# Transformers
log_scaler = Pipeline([
    ('log', FunctionTransformer(np.log1p, validate=True)),
    ('scale', StandardScaler())
])

scale_only = Pipeline([
    ('scale', StandardScaler())
])

# ColumnTransformer to handle different transformations
preprocessor = ColumnTransformer([
    ('log_scaled', log_scaler, right_skewed),
    ('scaled', scale_only, bounded_features)
])

X_scaled_array = preprocessor.fit_transform(df[right_skewed + bounded_features])

feature_names = right_skewed + bounded_features

X_scaled = pd.DataFrame(X_scaled_array, columns=feature_names, index=df.index)


In [ ]:
X_scaled.hist(figsize=(20,20))

In [ ]:
K = range(2, 11)
inertia = []
models  = []

for k in K:
    kmeans = KMeans(n_clusters=k)
    models.append(kmeans)
    kmeans.fit(X_scaled)
    inertia.append(kmeans.inertia_)

# Plot elbow
plt.figure(figsize=(8,6))
plt.plot(K, inertia, 'o-', color='blue')
plt.xlabel('Number of clusters')
plt.ylabel('Inertia')
plt.title('Elbow Method For Optimal k')
plt.show()

Let's try silhuoette score.

In [ ]:



sil_scores = []
K = range(len(models))
for i in K:
    kmeans = models[i]
    labels = kmeans.fit_predict(X_scaled)
    sil_scores.append(silhouette_score(X_scaled, labels))

plt.plot(K, sil_scores, 'o-')
plt.xlabel('Number of clusters')
plt.ylabel('Silhouette Score')
plt.title('Silhouette Analysis')
plt.show()


### Take K = 7

In [ ]:

# KMeans for 7 clusters
kmeans = models[5]
df['cluster'] = kmeans.fit_predict(X_scaled)


In [ ]:

# t-SNE on scaled features
tsne = TSNE(n_components=2, perplexity=50, n_iter=1000)
X_embedded = tsne.fit_transform(X_scaled)

# Plot
plt.figure(figsize=(10,7))
sns.scatterplot(
    x=X_embedded[:,0], y=X_embedded[:,1],
    hue=df['cluster'],
    palette='tab10',
    s=60,
    alpha=0.8
)
plt.title('t-SNE visualization of 7 KMeans clusters')
plt.xlabel('t-SNE 1')
plt.ylabel('t-SNE 2')
plt.legend(title='Cluster')
plt.show()


#Let's try DBScan

In [ ]:

dbscan = DBSCAN(eps=1.3, min_samples=10)  # adjust eps/min_samples
dbscan_labels = dbscan.fit_predict(X_scaled)

df['dbscan_cluster'] = dbscan_labels

#DBScan is Obviously bad.

In [ ]:
tsne = TSNE(n_components=2, perplexity=30)
tsne_results = tsne.fit_transform(X_scaled)

df['tsne_1'] = tsne_results[:,0]
df['tsne_2'] = tsne_results[:,1]
plt.figure(figsize=(10,7))
palette = sns.color_palette("tab10", np.unique(dbscan_labels).size)
sns.scatterplot(
    x='tsne_1', y='tsne_2',
    hue='dbscan_cluster',
    palette=palette,
    data=df,
    legend='full',
    s=50
)
plt.title("t-SNE Visualization of DBSCAN Clusters")
plt.xlabel("t-SNE 1")
plt.ylabel("t-SNE 2")
plt.legend(title="DBSCAN Cluster")
plt.show()

#Lets try GMM

In [ ]:
n_clusters = 7

gmm = GaussianMixture(n_components=n_clusters, covariance_type='full',n_init=100)
gmm.fit(X_scaled)

# Assign cluster based on highest probability
gmm_labels = gmm.predict(X_scaled)
df['gmm_cluster'] = gmm_labels

tsne = TSNE(n_components=2, perplexity=30)
tsne_results = tsne.fit_transform(X_scaled)

df['tsne_1'] = tsne_results[:,0]
df['tsne_2'] = tsne_results[:,1]

plt.figure(figsize=(10,7))
palette = sns.color_palette("tab10", n_clusters)
sns.scatterplot(
    x='tsne_1', y='tsne_2',
    hue='gmm_cluster',
    palette=palette,
    data=df,
    legend='full',
    s=50
)
plt.title("t-SNE Visualization of GMM Clusters")
plt.xlabel("t-SNE 1")
plt.ylabel("t-SNE 2")
plt.legend(title="GMM Cluster")
plt.show()

#GMM Looks slightly better than Kmeans on the t-SNE plot, let's check summary table and see if it makes business sense.

In [ ]:
all_features = [
    'BALANCE', 'BALANCE_FREQUENCY', 'PURCHASES', 'ONEOFF_PURCHASES',
    'INSTALLMENTS_PURCHASES', 'CASH_ADVANCE', 'PURCHASES_FREQUENCY',
    'ONEOFF_PURCHASES_FREQUENCY', 'PURCHASES_INSTALLMENTS_FREQUENCY',
    'CASH_ADVANCE_FREQUENCY', 'CASH_ADVANCE_TRX', 'PURCHASES_TRX',
    'CREDIT_LIMIT', 'PAYMENTS', 'MINIMUM_PAYMENTS', 'PRC_FULL_PAYMENT', 'TENURE'
]

cluster_summary_gmm = df.groupby('gmm_cluster')[all_features].agg(['mean', 'median']).round(2)
cluster_summary_gmm

##Cluster Number 4 is the most Dominant.

In [ ]:


palette = sns.color_palette("Set2", df['gmm_cluster'].nunique())

fig, ax = plt.subplots(figsize=(8, 5))
sns.countplot(
    data=df,
    x='gmm_cluster',
    palette=palette,
    order=sorted(df['gmm_cluster'].unique()),
    ax=ax
)

ax.set_title('Cluster Sizes (GMM)', fontsize=16, fontweight='bold', pad=15)
ax.set_xlabel('Cluster', fontsize=13)
ax.set_ylabel('Number of Customers', fontsize=13)

for p in ax.patches:
    ax.annotate(
        f'{int(p.get_height()):,}',
        (p.get_x() + p.get_width() / 2., p.get_height()),
        ha='center', va='bottom', fontsize=12, fontweight='bold'
    )

sns.despine()
plt.tight_layout()
plt.show()

#Pick the Cluster Number from the Drop Down List in the Dashboard to see Its characteristics.

In [ ]:


# ── Config ────────────────────────────────────────────────────────────────────
FEATURES = [
    'BALANCE', 'BALANCE_FREQUENCY', 'PURCHASES', 'ONEOFF_PURCHASES',
    'INSTALLMENTS_PURCHASES', 'CASH_ADVANCE', 'PURCHASES_FREQUENCY',
    'ONEOFF_PURCHASES_FREQUENCY', 'PURCHASES_INSTALLMENTS_FREQUENCY',
    'CASH_ADVANCE_FREQUENCY', 'CASH_ADVANCE_TRX', 'PURCHASES_TRX',
    'CREDIT_LIMIT', 'PAYMENTS', 'MINIMUM_PAYMENTS', 'PRC_FULL_PAYMENT', 'TENURE'
]

PALETTE  = sns.color_palette("Set2", df['gmm_cluster'].nunique())
BG_COLOR = '#F7F9FC'
clusters = sorted(df['gmm_cluster'].unique())

# ── Widgets ───────────────────────────────────────────────────────────────────
dropdown = widgets.Dropdown(
    options=[(f'Cluster {c}', c) for c in clusters],
    description='Cluster:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='200px')
)

out = widgets.Output()

# ── Plot function ─────────────────────────────────────────────────────────────
def plot_cluster(cluster_id):
    with out:
        clear_output(wait=True)

        sub   = df[df['gmm_cluster'] == cluster_id]
        color = PALETTE[clusters.index(cluster_id)]
        n     = len(sub)
        pct   = n / len(df) * 100

        fig = plt.figure(figsize=(22, 26), facecolor=BG_COLOR)
        fig.suptitle(
            f'Cluster {cluster_id}  ·  {n:,} customers  ({pct:.1f}% of total)',
            fontsize=20, fontweight='bold', y=0.98, color='#2d2d2d'
        )

        gs = gridspec.GridSpec(
            5, 4, figure=fig,
            hspace=0.55, wspace=0.35,
            top=0.94, bottom=0.04, left=0.06, right=0.97
        )

        # ── Row 0-3: distribution plots for each feature ──────────────────────
        for i, feat in enumerate(FEATURES):
            row, col = divmod(i, 4)
            ax = fig.add_subplot(gs[row, col])
            ax.set_facecolor(BG_COLOR)

            sns.histplot(sub[feat], kde=True, color=color, alpha=0.6,
                         line_kws={'lw': 2}, ax=ax)
            sns.kdeplot(df[feat], color='#aaaaaa', lw=1.2,
                        linestyle='--', ax=ax, label='All')

            mean_val   = sub[feat].mean()
            median_val = sub[feat].median()
            ax.axvline(mean_val,   color='#e74c3c', lw=1.5, linestyle='-',  label=f'μ {mean_val:.1f}')
            ax.axvline(median_val, color='#2c3e50', lw=1.5, linestyle=':', label=f'M {median_val:.1f}')

            ax.set_title(feat, fontsize=9, fontweight='bold', color='#2d2d2d')
            ax.set_xlabel('')
            ax.tick_params(labelsize=7)
            ax.legend(fontsize=6, loc='upper right', frameon=False)
            sns.despine(ax=ax)



        # ── Row 4 col 1-2: radar / spider chart ───────────────────────────────
        ax_radar = fig.add_subplot(gs[4, 1:3], polar=True)
        radar_feats = [
            'BALANCE', 'PURCHASES', 'CASH_ADVANCE', 'CREDIT_LIMIT',
            'PAYMENTS', 'PRC_FULL_PAYMENT', 'PURCHASES_FREQUENCY', 'CASH_ADVANCE_FREQUENCY'
        ]
        norm = lambda f: (sub[f].mean() - df[f].min()) / (df[f].max() - df[f].min() + 1e-9)
        values = [norm(f) for f in radar_feats]
        values += values[:1]

        angles = np.linspace(0, 2 * np.pi, len(radar_feats), endpoint=False).tolist()
        angles += angles[:1]

        ax_radar.plot(angles, values, color=color, lw=2)
        ax_radar.fill(angles, values, color=color, alpha=0.25)
        ax_radar.set_xticks(angles[:-1])
        ax_radar.set_xticklabels(
            [f.replace('_', '\n') for f in radar_feats],
            fontsize=7, color='#2d2d2d'
        )
        ax_radar.set_yticklabels([])
        ax_radar.set_title('Feature Profile\n(normalized vs global range)\n',
                            fontsize=9, fontweight='bold', pad=18)



        # ── Row 4 col 3: stats table ──────────────────────────────────────────
        ax_tbl = fig.add_subplot(gs[4, 3])
        ax_tbl.axis('off')
        key_feats = ['BALANCE', 'PURCHASES', 'CREDIT_LIMIT',
                     'PAYMENTS', 'CASH_ADVANCE', 'PRC_FULL_PAYMENT']
        tbl_data = [[f, f'{sub[f].mean():.1f}', f'{sub[f].median():.1f}'] for f in key_feats]
        tbl = ax_tbl.table(
            cellText=tbl_data,
            colLabels=['Feature', 'Mean', 'Median'],
            cellLoc='center', loc='center',
            bbox=[0, 0.1, 1, 0.85]
        )
        tbl.auto_set_font_size(False)
        tbl.set_fontsize(8)
        for (r, c_), cell in tbl.get_celld().items():
            cell.set_edgecolor('#cccccc')
            if r == 0:
                cell.set_facecolor(color)
                cell.set_text_props(color='white', fontweight='bold')
            else:
                cell.set_facecolor('#ffffff' if r % 2 == 0 else '#f2f2f2')
        ax_tbl.set_title('Key Stats', fontsize=9, fontweight='bold')

        plt.show()

# ── Wire up & display ─────────────────────────────────────────────────────────
def on_change(change):
    if change['name'] == 'value':
        plot_cluster(change['new'])

dropdown.observe(on_change, names='value')

display(widgets.VBox([
    widgets.HTML("<h3 style='font-family:sans-serif;color:#2d2d2d'>🗂️ GMM Cluster Explorer</h3>"),
    dropdown,
    out
]))

plot_cluster(clusters[0])

| Cluster | Label | Risk Level | Core Action |
|---|---|---|---|
| 0 | Pure High Spenders | Low | Retain, upsell premium rewards |
| 1 | Dual Users | Medium | Divert cash advance to personal credit line |
| 2 | Installment Loyalists | Low | Retail instalment partnerships |
| 3 | Occasional Splurgers | Low–Medium | Seasonal campaigns + introduce instalments |
| 4 | Pure Cash Advance | High | Debt relief products + risk monitoring |
| 5 | Maximum Revolvers | High | Balance transfer + relationship management |
| 6 | Cash Advance + Light Buyer | High | Refinancing + purchase behaviour nudges |

#Detailed Explanation in PDF.